In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import pickle

df = pd.read_csv('features.csv')
print(df.shape)
df.head()

(4647, 9)


,temperature,humidity,motion,power,hour,temp_diff,power_diff,temp_deviation,anomaly_injected
0,23.49,34.17,NaN,NaN,19,0.00,0.00,0.000,0
1,45.23,54.95,0.0,489.32,19,21.74,0.00,10.870,0
2,20.11,66.75,0.0,407.93,19,-25.12,-81.39,-9.500,0
3,19.17,60.97,0.0,70.15,19,-0.94,-337.78,-7.830,0
4,20.46,64.67,0.0,443.74,19,1.29,373.59,-5.232,0


In [5]:
df = df.dropna()
print(df.shape)

X = df[['temperature', 'humidity', 'motion', 'power',
        'hour', 'temp_diff', 'power_diff', 'temp_deviation']]
y = df['anomaly_injected']

print(f"Features shape: {X.shape}")
print(f"Anomaly count: {y.sum()}")

(4646, 9)
Features shape: (4646, 8)
Anomaly count: 693


In [6]:
model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

model.fit(X)
print("Model trained.")

Model trained.


In [7]:
preds = model.predict(X)

# Isolation Forest outputs 1 for normal, -1 for anomaly
# Convert to match our labels: 1 = anomaly, 0 = normal
preds_binary = (preds == -1).astype(int)

print(confusion_matrix(y, preds_binary))
print()
print(classification_report(y, preds_binary, target_names=['Normal', 'Anomaly']))

[[3900   53]
 [ 513  180]]

              precision    recall  f1-score   support

      Normal       0.88      0.99      0.93      3953
     Anomaly       0.77      0.26      0.39       693

    accuracy                           0.88      4646
   macro avg       0.83      0.62      0.66      4646
weighted avg       0.87      0.88      0.85      4646



In [8]:
model2 = IsolationForest(
    n_estimators=100,
    contamination=0.15,
    random_state=42
)

model2.fit(X)
preds2 = model2.predict(X)
preds2_binary = (preds2 == -1).astype(int)

print(confusion_matrix(y, preds2_binary))
print()
print(classification_report(y, preds2_binary, target_names=['Normal', 'Anomaly']))

[[3680  273]
 [ 269  424]]

              precision    recall  f1-score   support

      Normal       0.93      0.93      0.93      3953
     Anomaly       0.61      0.61      0.61       693

    accuracy                           0.88      4646
   macro avg       0.77      0.77      0.77      4646
weighted avg       0.88      0.88      0.88      4646



In [9]:
with open('anomaly_model.pkl', 'wb') as f:
    pickle.dump(model2, f)

print("Model saved as anomaly_model.pkl")

Model saved as anomaly_model.pkl


In [10]:
feature_map = {
    'temperature': ['temperature', 'temp_diff', 'temp_deviation', 'hour'],
    'humidity':    ['humidity', 'hour'],
    'motion':      ['motion', 'hour'],
    'power':       ['power', 'power_diff', 'hour'],
}

models_per_sensor = {
    sensor: IsolationForest(n_estimators=100, contamination=0.15, random_state=42)
    for sensor in feature_map
}

for sensor, feat_cols in feature_map.items():
    models_per_sensor[sensor].fit(X[feat_cols])

scores = pd.DataFrame()
for sensor, feat_cols in feature_map.items():
    preds = models_per_sensor[sensor].predict(X[feat_cols])
    scores[sensor] = (preds == -1).astype(int)

scores['final'] = (scores.sum(axis=1) >= 1).astype(int)

print(confusion_matrix(y, scores['final']))
print()
print(classification_report(y, scores['final'], target_names=['Normal', 'Anomaly']))

[[2647 1306]
 [  80  613]]

              precision    recall  f1-score   support

      Normal       0.97      0.67      0.79      3953
     Anomaly       0.32      0.88      0.47       693

    accuracy                           0.70      4646
   macro avg       0.65      0.78      0.63      4646
weighted avg       0.87      0.70      0.74      4646



In [11]:
with open('anomaly_models_per_sensor.pkl', 'wb') as f:
    pickle.dump((models_per_sensor, feature_map), f)

print("Saved anomaly_models_per_sensor.pkl")

Saved anomaly_models_per_sensor.pkl


In [ ]:
scores['final_v2'] = (scores[['temperature','humidity','motion','power']].sum(axis=1) >= 2).astype(int)

print(confusion_matrix(y, scores['final_v2']))
print()
print(classification_report(y, scores['final_v2'], target_names=['Normal', 'Anomaly']))